In [1]:
import sys
# Ensure we aren't accidentally pulling from Roaming
sys.path = [p for p in sys.path if "Roaming" not in p]

try:
    from scipy import sparse
    import sklearn
    print("Success! Scipy Sparse and Sklearn are both loaded.")
except ImportError as e:
    print(f"Still failing: {e}")

Success! Scipy Sparse and Sklearn are both loaded.


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
#from langchain_community.chains import LLMChain
import json
from langchain_community.llms import Ollama
import os
import urllib.request
from pathlib import Path
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
#from langchain.evaluation import load_evaluator
from datasets import Dataset
from ragas.metrics.collections import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from datasets import load_dataset
from langchain_core.prompts import ChatPromptTemplate
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics.base import Metric


load_dotenv()

c:\Users\Public\anaconda3\envs\agenticEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_3428\4194381195.py:31: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_3428\4194381195.py:31: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Fait

True

In [3]:
openai_api_key = os.getenv("OPENAI_API_KEY")

In [4]:
os.makedirs("../data", exist_ok=True)
urllib.request.urlretrieve("https://arxiv.org/pdf/2502.17429?", "../data/incremental_learning_semantic_seg.pdf")
pdf = Path("../data/incremental_learning_semantic_seg.pdf")
print(pdf.exists(), pdf.stat().st_size)

True 2342202


In [5]:
loader = PyPDFLoader("../data/incremental_learning_semantic_seg.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 700, chunk_overlap = 100)
chunks = splitter.split_documents(documents=documents)
print(f"Total Chunks Created : {len(chunks)}")

Total Chunks Created : 117


In [ ]:
# llm = Ollama(
#     model="qwen2.5",
#     temperature=0.2,
#     top_p=0.9,
#     num_ctx=4096
# )

# Model used to generate the synthetic dataset:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1, api_key=openai_api_key)

In [ ]:
# Prompt to generate synthetic dataset:
rag_prompt = PromptTemplate(
    input_variables=["context"],
    template="""
You are a teacher/professor.

Given the context below:
- Generate ONE high-quality, non-trivial question
- The question MUST be answerable ONLY from the context
- Provide a concise, factual answer
- Do NOT use external knowledge

Context:
{context}

Return JSON:
{{
  "question": "...",
  "answer": "...",
  "context": "..."
}}
"""
)

rag_chain = rag_prompt | llm

In [8]:
dataset = []

def create_dataset(chunks, rag_chain):
    dataset = []
    
    for chunk in chunks:
        
        response = rag_chain.invoke(input={
            'context' : chunk.page_content
        })

        try:
            res_text = json.loads(response.content)
            dataset.append(res_text)
            
        except Exception as e:
            print(f"Error : {str(e)}")

    return dataset

In [9]:
qa_dataset = create_dataset(chunks, rag_chain=rag_chain)

Error : Invalid control character at: line 4 column 142 (char 291)


In [10]:
qa_dataset

[{'question': 'What is the main focus of the research presented in the paper by Thengane et al.?',
  'answer': 'The main focus of the research is on Class-Incremental Imbalanced 3D Instance Segmentation (CLIMB-3D).',
  'context': 'arXiv:2502.17429v3  [cs.CV]  21 Nov 2025 THENGANE ET AL.: CLIMB-3D1 CLIMB-3D: Class-Incremental Imbalanced 3D Instance Segmentation Vishal Thengane1 v.thengane@surrey.ac.uk Jean Lahoud2 jean.lahoud@mbzuai.ac.ae Hisham Cholakkal2 hisham.cholakkal@mbzuai.ac.ae Rao Muhammad Anwer2 rao.anwer@mbzuai.ac.ae Lu Yin1 l.yin@surrey.ac.uk Xiatian Zhu1 xiatian.zhu@surrey.ac.uk Salman Khan2, 3 salman.khan@mbzuai.ac.ae 1 University of Surrey, Guildford, UK 2 Mohamed bin Zayed University of Artificial Intelligence, Abu Dhabi, UAE 3 Australian National University, Canberra, Australia Abstract While 3D instance segmentation (3DIS) has advanced significantly, most existing'},
 {'question': 'What is the main limitation of existing methods in 3D instance segmentation according to

In [11]:
len(qa_dataset[:10])

10

In [12]:
train_examples, test_examples = train_test_split(
    qa_dataset,
    test_size=0.2,          # 20% held out
    random_state=42,        # for reproducibility
    shuffle=True
)

print(f"Training on {len(train_examples)} examples, testing on {len(test_examples)} examples")

Training on 92 examples, testing on 24 examples


In [13]:
test_dataset = pd.DataFrame.from_records(data=test_examples)
test_dataset.head()

,question,answer,context
0,What is the main focus of the paper by Saining...,The main focus of the paper by Saining Xie et ...,"16THENGANE ET AL.: CLIMB-3D [71] Zifeng Wang, ..."
1,What new object categories are introduced in T...,"Pillow, Coffee Table, and Sofa Chair.",2THENGANE ET AL.: CLIMB-3D 1 Introduction Tabl...
2,What metric is used to assess the model's abil...,Forgetting Percentage Points (FPP),we report the mean Intersection over Union (mI...
3,What dataset is used to evaluate CLIMB-3D in t...,ScanNet200,These incremental scenarios are designed to pr...
4,What is one of the main contributions of Theng...,A novel problem setting of imbalanced class-in...,"THENGANE ET AL.: CLIMB-3D3 In summary, our con..."


In [17]:
test_dataset.to_csv("../data/test_dataset.csv", index=False)

In [14]:
train_examples[0]

{'question': 'What dataset is mentioned in the context that enables efforts similar to long-tailed recognition in 3D?',
 'answer': 'ScanNet200',
 'context': 'offers another direction, using class-based [6, 15, 23, 32, 33, 69] or per-example [45, 55, 62] adjustments to ensure fair contribution. Parameter regularisation improves generalisation through weight constraints [2], albeit requiring careful tuning. Other approaches leverage transfer learning [77, 81], self-supervision [41, 75], or contrastive learning [34, 41, 82] to improve rare-class representations. Long-tailed recognition is well-studied in 2D with large-scale datasets [15, 49, 67]; in 3D, ScanNet200 [16, 57] enables similar efforts. Prior 3D work focused on re-weighting, re-sampling, and transfer learning [57], while regularisation.'}

In [33]:
def create_train_dataset(train_examples, out_path):
    
    with open(out_path, "w+", encoding="utf8") as f:
        for example in train_examples:
            content = {
                "messages" : [
                    {"role":"user", "content": example['question']},
                    {"role":"assistant", "content":example['answer']}
                ]
            }
            f.write(json.dumps(content, ensure_ascii=False) +"\n")
    return "Train Dataset Created"

In [35]:
create_train_dataset(train_examples, "../data/train_dataset.jsonl")

'Train Dataset Created'

In [42]:
train_dataset = load_dataset("json", data_files="../data/train_dataset.jsonl")
print(train_dataset)
print(train_dataset['train'][:5])

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 92
    })
})
{'messages': [[{'role': 'user', 'content': 'What dataset is mentioned in the context that enables efforts similar to long-tailed recognition in 3D?'}, {'role': 'assistant', 'content': 'ScanNet200'}], [{'role': 'user', 'content': 'What is the title of the work by Li Jiang et al. presented at CVPR in 2020?'}, {'role': 'assistant', 'content': 'Pointgroup: Dual-set point grouping for 3d instance segmentation.'}], [{'role': 'user', 'content': 'What is the purpose of using the pseudo-label predictions of the frozen model Φ t−1 in the context of class imbalance?'}, {'role': 'assistant', 'content': 'The purpose is to leverage the pseudo-label predictions as a proxy for the class distribution across previously learned categories, allowing for the accumulation of class-wise frequency statistics for those classes.'}], [{'role': 'user', 'content': 'What are the two additional components introduced to 

RAG Evaluation

In [15]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_10772\2083950908.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [17]:
test_examples[0]

{'question': 'What is the main focus of the paper by Saining Xie et al. presented at ECCV?',
 'answer': 'The main focus of the paper by Saining Xie et al. is on unsupervised pre-training for 3D point cloud understanding.',
 'context': '16THENGANE ET AL.: CLIMB-3D [71] Zifeng Wang, Tong Jian, Kaushik Chowdhury, Yanzhi Wang, Jennifer Dy, and Stratis Ioannidis. Learn-prune-share for lifelong learning. In2020 IEEE International Conference on Data Mining (ICDM), pages 641–650. IEEE, 2020. [72] Saining Xie, Jiatao Gu, Demi Guo, Charles R Qi, Leonidas Guibas, and Or Litany. Pointcontrast: Unsupervised pre-training for 3d point cloud understanding. InECCV, pages 574–591. Springer, 2020. [73] Bo Yang, Jianan Wang, Ronald Clark, Qingyong Hu, Sen Wang, Andrew Markham, and Niki Trigoni. Learning object bounding boxes for 3d instance segmentation on point clouds.NeurIPS, 32, 2019.'}

In [21]:
# Model we are aiming to fine-tune:
from langchain_community.chat_models import ChatOllama
ft_model = ChatOllama(
    model="llama3.2:1b",
    temperature=0.0
)

In [ ]:
# Prompt used to get answers on the model that we are aiming to fine-tune:
ft_prompt = ChatPromptTemplate.from_template(
    template="""
You are answering a question using retrieved context from a document.

Instructions:
- Use ONLY the information in the context.
- You MAY paraphrase, summarize, and combine information.
- If the context provides partial information, answer as fully as possible.
- Do NOT introduce facts not supported by the context.
- Only say "I don't know" if the context gives no useful information at all.

Context:
{context}

Question:
{question}
"""
)

ft_chain = ft_prompt | ft_model

In [44]:
def generate_answers(dataset, chain):
    test_df = []
    for example in dataset:
        res = chain.invoke(input={
            'question': example['question'],
            'context': example['context']
        })
        response = res.content
        obj = {
            'question' : example['question'],
            'contexts' : example['context'],
            'ground_truth' : example['answer'],
            'base_model_answer' : response
        }
        test_df.append(obj)

    test_generated_dataset = pd.DataFrame.from_records(data=test_df)
    return test_generated_dataset
        

In [45]:
test_generated_dataset = generate_answers(test_examples, ft_chain)
test_generated_dataset.head()

,question,contexts,ground_truth,base_model_answer
0,What is the main focus of the paper by Saining...,"16THENGANE ET AL.: CLIMB-3D [71] Zifeng Wang, ...",The main focus of the paper by Saining Xie et ...,"The main focus of the paper ""Pointcontrast: Un..."
1,What new object categories are introduced in T...,2THENGANE ET AL.: CLIMB-3D 1 Introduction Tabl...,"Pillow, Coffee Table, and Sofa Chair.",I don't know.
2,What metric is used to assess the model's abil...,we report the mean Intersection over Union (mI...,Forgetting Percentage Points (FPP),To answer this question based on the provided ...
3,What dataset is used to evaluate CLIMB-3D in t...,These incremental scenarios are designed to pr...,ScanNet200,The dataset used to evaluate CLIMB-3D in the e...
4,What is one of the main contributions of Theng...,"THENGANE ET AL.: CLIMB-3D3 In summary, our con...",A novel problem setting of imbalanced class-in...,One of the main contributions of Thengane et a...


In [52]:
test_generated_dataset.rename(columns={'base_model_answer':'answer'},inplace=True)

In [54]:
test_generated_dataset.to_csv("../data/testEvalData_baseModel.csv", index=False)

In [3]:
test_generated_dataset = pd.read_csv("../data/testEvalData_baseModel.csv")

In [ ]:
def get_rag_evaluation(embed_name, dataset, model_name):
    
    
    ragas_data = []
    for _, row in dataset.iterrows():
        
        obj ={
            'question' : row['question'],
            'contexts' : [row['contexts']],
            'ground_truth' : row['ground_truth'],
            'answer' : row['answer']
            
        }
        ragas_data.append(obj)
    
    data = Dataset.from_list(ragas_data)

    # 1. Setup an OpenAI-compatible client for Ollama
    # Ollama provides an OpenAI-compatible endpoint at /v1 --> If any other model to be used here:
    client = OpenAI()

    # 2. Use the llm_factory instead of LangchainLLMWrapper
    # Use provider="openai" to use the compatible client
    #LLM given here is the judge. So, use the same LLM that is used to generate this synthetic data
    evaluator_llm = llm_factory(model_name, client=client)

    #Throwing some errors so switched to LangchainEmbeddingsWrapper:
    # evaluator_embeddings = embedding_factory("openai", model=embed_name, client=client)

    lc_embeddings = OllamaEmbeddings(
    model=embed_name  # e.g. "nomic-embed-text"
    )
    ragas_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)
    
    #Defining the metrics
    m1 = Faithfulness(llm=evaluator_llm)
    m2 = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
    m3 = AnswerCorrectness(llm=evaluator_llm, embeddings=ragas_embeddings)
    m4 = ContextPrecision(llm=evaluator_llm)

    metric_list = [m1, m2, m3, m4]
    
    results = evaluate(dataset=data,
                       metrics=metric_list,
                       embeddings=ragas_embeddings,
                        )
    eval_df = results.to_pandas()
    return eval_df

In [11]:
eval_df = get_rag_evaluation(embed_name="nomic-embed-text", dataset=test_generated_dataset, 
                             model_name="gpt-4o-mini") #Using the same model that we used earlier to generate the synthetic data
eval_df.head()

C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_3428\1015740776.py:25: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = embedding_factory("openai", model=embed_name, client=client)
C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_3428\1015740776.py:30: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)
Evaluating:   0%|          | 0/96 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM retur

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,answer_correctness,context_precision
0,What is the main focus of the paper by Saining...,"[16THENGANE ET AL.: CLIMB-3D [71] Zifeng Wang,...","The main focus of the paper ""Pointcontrast: Un...",The main focus of the paper by Saining Xie et ...,0.666667,0.000000,0.838266,1.0
1,What new object categories are introduced in T...,[2THENGANE ET AL.: CLIMB-3D 1 Introduction Tab...,I don't know.,"Pillow, Coffee Table, and Sofa Chair.",0.000000,0.000000,0.147293,1.0
2,What metric is used to assess the model's abil...,[we report the mean Intersection over Union (m...,To answer this question based on the provided ...,Forgetting Percentage Points (FPP),1.000000,0.697478,0.959224,1.0
3,What dataset is used to evaluate CLIMB-3D in t...,[These incremental scenarios are designed to p...,The dataset used to evaluate CLIMB-3D in the e...,ScanNet200,1.000000,0.866720,0.947993,1.0
4,What is one of the main contributions of Theng...,"[THENGANE ET AL.: CLIMB-3D3 In summary, our co...",One of the main contributions of Thengane et a...,A novel problem setting of imbalanced class-in...,1.000000,0.806459,0.830514,1.0


In [18]:
mean_faithfulness_score = eval_df['faithfulness'].mean()
mean_answer_relevancy_score = eval_df['answer_relevancy'].mean()
mean_answer_correctness_score = eval_df['answer_correctness'].mean()
print(mean_faithfulness_score, mean_answer_relevancy_score, mean_answer_correctness_score)

0.6577380952380952 0.6159300899170209 0.6665186637507786


In [14]:
eval_df.to_csv('../data/baseModel_evaluationScores.csv', index=False)